# Data Structures & Algorithms — 30 illustrated Colab experiments

Companion to the course guide: [DATA_STRUCTURES_ALGORITHMS.md](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md). Each section has three parts:

1. **Concept** — a short recap and a link to the full lesson.
2. **See it** — a small, runnable demonstration that draws or traces the structure.
3. **Measure it** — the experiment: a chart plus a CSV in `dsa_results/`.

Run **Runtime → Run all** (about one minute). Before each *Measure it* cell, write down your prediction. Counts (comparisons, moves, swaps) are exact; times (ms, µs) are medians and vary between runs and machines.

In [ ]:
# Setup: use the repository copy if present, otherwise download the lab module from GitHub.
import importlib.util, pathlib, sys, urllib.request
from pathlib import Path
candidates = [Path('dsa_lab/experiments.py'), Path('experiments.py')]   # repo root, or inside dsa_lab/
path = next((p for p in candidates if p.exists()), Path('dsa_experiments.py'))
if not path.exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/FUllSCIPTS/ATLAS/main/dsa_lab/experiments.py', path)
spec = importlib.util.spec_from_file_location('dsa', path)
dsa = importlib.util.module_from_spec(spec); sys.modules['dsa'] = dsa; spec.loader.exec_module(dsa)
dsa.self_test()

import matplotlib.pyplot as plt
from IPython.display import Image, display
OUT = Path('dsa_results'); OUT.mkdir(exist_ok=True)

def experiment(name):
    rows = dsa.run_experiment(name)
    png = dsa.save_plot(name, rows, OUT)
    ident, title, labels, *_ = dsa.EXPERIMENTS[name]
    print(f'{ident} {title}')
    print('  columns:', ', '.join(labels))
    for r in rows:
        print('  ', r[0], '->', ', '.join(f'{v:.4g}' if isinstance(v, float) else str(v) for v in r[1:]))
    display(Image(filename=str(png)))
    return rows

def show(kind, values, title):
    fig = dsa.draw_structure(kind, values, title=title); plt.show()

## E01 — Growth rates

**Concept.** Counting is done on the **code**, not the clock. A triangular double loop costs n(n−1)/2. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-2--counting-operations-and-measuring-honestly).

**See it** (run the next cell).

In [ ]:
def triangle(n):
    return sum(1 for i in range(n) for j in range(i))
for n in (5, 10, 20, 40):
    print(f'n={n:>3}  triangle={triangle(n):>4}  n(n-1)/2={n*(n-1)//2:>4}  ratio to previous n: {triangle(n)/max(1,triangle(n//2)):.2f}')

**Measure it.** Predict the chart first, then run.

*What to look for:* Doubling n multiplies n² by 4 and n log n by a little more than 2. On a log scale these become straight lines of different slope.

In [ ]:
rows_E01 = experiment('growth')

## E02 — Linear vs binary search

**Concept.** Binary search halves a **sorted** interval; linear search walks every item. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-19--searching).

**See it** (run the next cell).

In [ ]:
a = [2, 4, 6, 8, 10, 12, 14]
lo, hi = 0, len(a) - 1
while lo <= hi:                      # trace the interval for target 12
    mid = (lo + hi) // 2
    print(f'lo={lo} hi={hi} mid={mid} a[mid]={a[mid]}')
    if a[mid] == 12: print('found at', mid); break
    lo, hi = (mid + 1, hi) if a[mid] < 12 else (lo, mid - 1)

**Measure it.** Predict the chart first, then run.

*What to look for:* The binary column equals ⌊log₂ n⌋ + 1 exactly for an absent key larger than every item.

In [ ]:
rows_E02 = experiment('search')

## E03 — Best, average and worst case

**Concept.** One algorithm has several cost functions. Linear search: 1 (first item), (n+1)/2 (average successful), n (absent). [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-3--asymptotic-notation-cases-amortised-cost-and-recurrences).

**See it** (run the next cell).

In [ ]:
a = list(range(10))
print('comparisons to find each key:', [dsa.linear_search(a, x)[1] for x in a])
print('mean =', sum(dsa.linear_search(a, x)[1] for x in a) / len(a), '= (n+1)/2 =', (len(a)+1)/2)

**Measure it.** Predict the chart first, then run.

*What to look for:* Best is flat, average and worst are both linear — but with different constants.

In [ ]:
rows_E03 = experiment('search_cases')

## E04 — Bubble vs merge vs quick

**Concept.** Same shuffled input, three algorithms, exact comparison counts. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-20--sorting-stability-and-the-n-log-n-lower-bound).

**See it** (run the next cell).

In [ ]:
data = [7, 2, 6, 1]
print('merge sort:', dsa.merge_sort(data))
print('bubble sort:', dsa.bubble_sort(data))
print('quick (functional):', dsa.quick_sort(data))

**Measure it.** Predict the chart first, then run.

*What to look for:* Bubble grows like n²/2; merge like n log₂ n; the functional quicksort like ~1.4 n log₂ n (it compares each item with the pivot up to twice).

In [ ]:
rows_E04 = experiment('sort')

## E05 — Input order matters

**Concept.** Insertion sort does n−1 comparisons on sorted input and n(n−1)/2 on reversed input. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-20--sorting-stability-and-the-n-log-n-lower-bound).

**See it** (run the next cell).

In [ ]:
for name, data in (('sorted', [1, 2, 3, 4, 5]), ('reversed', [5, 4, 3, 2, 1])):
    print(name, dsa.insertion_sort(data))

**Measure it.** Predict the chart first, then run.

*What to look for:* The worst-case column equals n(n−1)/2 exactly; the random column is about half of it.

In [ ]:
rows_E05 = experiment('sort_cases')

## E06 — Measured time

**Concept.** Times are medians of five runs; they suggest, but cannot prove, growth rates. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-2--counting-operations-and-measuring-honestly).

**See it** (run the next cell).

In [ ]:
import time
t = time.perf_counter(); sum(range(10**6)); print('one run of sum(range(10**6)):', round(1000*(time.perf_counter()-t), 2), 'ms')
t = time.perf_counter(); sum(range(10**6)); print('second run:', round(1000*(time.perf_counter()-t), 2), 'ms  <- noise')

**Measure it.** Predict the chart first, then run.

*What to look for:* Re-run the cell: the curves wobble. Compare their *shape* with E02 and E04, not their absolute values.

In [ ]:
rows_E06 = experiment('runtime')

## E07 — Front insertion: array vs linked list

**Concept.** `list.insert(0, x)` shifts every element; a linked `push_front` changes two pointers. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-5--singly-linked-lists).

**See it** (run the next cell).

In [ ]:
show('singly', [7, 11, 19], 'push_front(7) onto 11 → 19: one new node, head moves')
s = dsa.SinglyLinkedList([11, 19]); s.push_front(7); print(s.to_list(), 'valid:', s.validate())

**Measure it.** Predict the chart first, then run.

*What to look for:* For small n the C memmove inside `list.insert` wins; past a few thousand items the quadratic total cost dominates. Constant factors vs growth rate.

In [ ]:
rows_E07 = experiment('array_linked')

## E08 — Stack and queue removals

**Concept.** Removing all n items: `list.pop()` and `deque.popleft()` are O(1) each; `list.pop(0)` is O(n) each. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-10--queues-deques-and-ring-buffers-circular-arrays).

**See it** (run the next cell).

In [ ]:
from collections import deque
stack, queue = [], deque()
for x in 'ABC': stack.append(x); queue.append(x)
print('stack pops:', [stack.pop() for _ in range(3)], '(LIFO)')
print('queue pops:', [queue.popleft() for _ in range(3)], '(FIFO)')

**Measure it.** Predict the chart first, then run.

*What to look for:* Only the `pop(0)` curve bends upwards. The input is rebuilt outside the timer in every repetition.

In [ ]:
rows_E08 = experiment('stack_queue')

## E09 — Lookups: dict vs scan vs bisect

**Concept.** n lookups each: dict Θ(1) expected, scan Θ(n), binary search Θ(log n). [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-11--hash-tables-chaining-open-addressing-load-factor-and-resizing).

**See it** (run the next cell).

In [ ]:
d = {'UCC': 0, 'Kent Station': 2}
print('dict lookup:', d['Kent Station'], '| hash bucket for an 8-slot table:', hash('Kent Station') % 8, '(varies per run: string hashing is randomised)')

**Measure it.** Predict the chart first, then run.

*What to look for:* The scan curve is quadratic in total (n lookups × n), bisect is n log n, dict is almost flat per lookup.

In [ ]:
rows_E09 = experiment('hash')

## E10 — Balanced vs skewed BST

**Concept.** Insert order decides the shape: middle-first gives height ⌊log₂ n⌋+1, ascending gives a chain. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-13--binary-search-trees-search-insert-and-the-three-deletion-cases).

**See it** (run the next cell).

In [ ]:
skew = None
for k in range(1, 8): skew = dsa.bst_insert_node(skew, k)
dsa.draw_tree(dsa.build_balanced(list(range(1, 8))), title='balanced: height 3'); plt.show()
dsa.draw_tree(skew, title='ascending inserts: height 7'); plt.show()

**Measure it.** Predict the chart first, then run.

*What to look for:* The skewed curve is the identity line; the balanced one tracks the log reference.

In [ ]:
rows_E10 = experiment('bst')

## E11 — BFS and DFS

**Concept.** Both examine each edge of the reachable part exactly once: Θ(V + E). [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-17--graphs-representations-bfs-and-dfs).

**See it** (run the next cell).

In [ ]:
g = {'A': ['B', 'C'], 'B': ['D'], 'C': ['D'], 'D': []}
print('BFS:', dsa.bfs(g, 'A'), ' DFS:', dsa.dfs(g, 'A'))
print('BFS distances:', dsa.bfs_distances(g, 'A')[0])

**Measure it.** Predict the chart first, then run.

*What to look for:* Edges examined ≤ E (only vertices reachable from 0 are explored); BFS and DFS examine the same edges in a different order.

In [ ]:
rows_E11 = experiment('graph')

## E12 — Dijkstra

**Concept.** Relax each edge once; push a heap entry only when a distance improves. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-18--shortest-paths-topological-order-union-find-and-spanning-trees).

**See it** (run the next cell).

In [ ]:
g = {'A': [('B', 2), ('C', 8)], 'B': [('C', 1)], 'C': []}
print(dsa.dijkstra(g, 'A'), dsa.dijkstra_path(g, 'A', 'C'))

**Measure it.** Predict the chart first, then run.

*What to look for:* Relaxations equal the edges out of reachable vertices; pushes are fewer because most relaxations do not improve a distance.

In [ ]:
rows_E12 = experiment('shortest_path')

## E13 — Fibonacci: recursion vs DP

**Concept.** Naive recursion recomputes subproblems; memoisation solves each once. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-22--greedy-algorithms-dynamic-programming-and-backtracking).

**See it** (run the next cell).

In [ ]:
for k in (5, 10, 15):
    print(k, 'naive calls:', dsa.fib_naive_calls(k)[1], ' memo calls:', dsa.fib_memo_calls(k)[1])

**Measure it.** Predict the chart first, then run.

*What to look for:* On the log scale the naive line is straight (exponential growth, ratio ≈ 1.618 per step); memo and loop are linear.

In [ ]:
rows_E13 = experiment('recursion_dp')

## E14 — Memory

**Concept.** `tracemalloc` reports Python allocations: a dict stores hashes, keys and values, so it needs more than a list. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-2--counting-operations-and-measuring-honestly).

**See it** (run the next cell).

In [ ]:
import sys
print('list of 1000 ints:', sys.getsizeof(list(range(1000))), 'bytes (container only)')
print('dict of 1000 ints:', sys.getsizeof({i: i for i in range(1000)}), 'bytes (container only)')

**Measure it.** Predict the chart first, then run.

*What to look for:* All three grow linearly; the slopes differ by structure overhead.

In [ ]:
rows_E14 = experiment('memory')

## E15 — Stability

**Concept.** A stable sort keeps equal keys in their input order. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-20--sorting-stability-and-the-n-log-n-lower-bound).

**See it** (run the next cell).

In [ ]:
recs = [(1, 'a'), (0, 'b'), (1, 'c'), (0, 'd')]
print('merge:', dsa.merge_sort(recs, key=lambda r: r[0])[0])
print('heap :', dsa.heap_sort(recs, key=lambda r: r[0])[0], '<- b/d or a/c may swap')

**Measure it.** Predict the chart first, then run.

*What to look for:* Merge and insertion sort stay at zero; heap sort's long-distance swaps break equal-key order.

In [ ]:
rows_E15 = experiment('sorting_stability')

## E16 — Singly linked chains

**Concept.** Each bucket is a singly linked list; new keys go to the head. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-11--hash-tables-chaining-open-addressing-load-factor-and-resizing).

**See it** (run the next cell).

In [ ]:
t = dsa.ChainHash(4, 'singly')
for k in (1, 5, 9, 3, 7): t.put(k, k * 10)
for i, root in enumerate(t.buckets):
    chain, p = [], root
    while p: chain.append(p.key); p = p.next
    print(f'bucket {i}:', ' -> '.join(map(str, chain)) or 'empty')

**Measure it.** Predict the chart first, then run.

*What to look for:* Mean comparisons follow 1 + α/2 with α = 4 here; the maximum chain stays small for well-spread keys.

In [ ]:
rows_E16 = experiment('hash_singly')

## E17 — Doubly linked chains

**Concept.** Deleting a found node only needs its neighbours; `validate()` checks every prev pointer. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-6--doubly-linked-lists).

**See it** (run the next cell).

In [ ]:
t = dsa.ChainHash(2, 'doubly')
for k in (0, 2, 4, 6): t.put(k, k)
t.delete(4); print('valid after deleting a middle node:', t.validate(), '| chain lengths', t.chain_lengths())

**Measure it.** Predict the chart first, then run.

*What to look for:* Same lookup cost as E16: the extra pointer speeds deletion of a known node, not search.

In [ ]:
rows_E17 = experiment('hash_doubly')

## E18 — Adversarial keys

**Concept.** Keys that are all multiples of m land in one bucket: the table becomes a linked list. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-11--hash-tables-chaining-open-addressing-load-factor-and-resizing).

**See it** (run the next cell).

In [ ]:
m = 8
t = dsa.chain_table(16, m, 'singly', keys=[i * m for i in range(16)])
print('chain lengths:', t.chain_lengths())

**Measure it.** Predict the chart first, then run.

*What to look for:* Uniform keys stay near the theory line; adversarial keys grow linearly (log scale!).

In [ ]:
rows_E18 = experiment('hash_load')

## E19 — Traversals with a stack and a queue

**Concept.** Depth-first orders use a stack; level order uses a queue. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-12--trees-and-the-four-traversals).

**See it** (run the next cell).

In [ ]:
root = None
for k in (8, 4, 12, 2, 6, 10, 14): root = dsa.bst_insert_node(root, k)
dsa.draw_tree(root, title='keys 8,4,12,2,6,10,14'); plt.show()
for f in (dsa.preorder_stack, dsa.inorder_stack, dsa.postorder_stack, dsa.levelorder_queue):
    print(f'{f.__name__:<18}', f(root))

**Measure it.** Predict the chart first, then run.

*What to look for:* All three are linear; differences are constant factors.

In [ ]:
rows_E19 = experiment('tree_stack_queue')

## E20 — AVL rotations

**Concept.** Ascending inserts would make a chain; AVL rotations keep the height logarithmic. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-14--avl-trees-and-the-four-rotations).

**See it** (run the next cell).

In [ ]:
avl = None
for k in range(1, 16): avl = dsa.avl_insert(avl, k)
dsa.draw_tree(avl, title='AVL after inserting 1..15 in order'); plt.show()
print('height', dsa.validate_avl(avl))

**Measure it.** Predict the chart first, then run.

*What to look for:* AVL search steps track log₂ n; the plain BST walks all n nodes.

In [ ]:
rows_E20 = experiment('avl')

## E21 — Circular singly linked list — Josephus

**Concept.** Only `tail` is stored; `tail.next` is the head; every k-th person leaves the ring. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-7--circular-singly-linked-lists).

**See it** (run the next cell).

In [ ]:
show('circular', [1, 2, 3, 4, 5, 6, 7], 'Josephus ring, n = 7')
order, moves = dsa.josephus_circular(7, 3)
print('elimination order:', order, '| pointer moves:', moves, '| survivor by formula:', dsa.josephus_formula(7, 3))
ring = dsa.CircularSinglyLinkedList([0, 1, 2]); ring.rotate(1)
print('after rotate(1):', ring.to_list(), '| tail.next is head:', ring.tail.next is ring.head)

**Measure it.** Predict the chart first, then run.

*What to look for:* Pointer moves equal n(k−1) exactly; list shifts grow quadratically.

In [ ]:
rows_E21 = experiment('circular_singly')

## E22 — Circular doubly list + sentinel — LRU cache

**Concept.** A dict finds the node, the circular list reorders it: both O(1). [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-8--circular-doubly-linked-lists-sentinels-lru-caches-and-cycle-detection).

**See it** (run the next cell).

In [ ]:
show('circular-doubly', [7, 11, 19], 'sentinel S joins both ends')
cd = dsa.CircularDoublyLinkedList([1, 2, 3, 4, 5])
print('rotate(2):', cd.rotate(2), 'steps ->', cd.to_list(), '| rotate(4):', cd.rotate(4), 'step ->', cd.to_list())
lru = dsa.LRUCache(2); lru.put('a', 1); lru.put('b', 2); lru.get('a'); lru.put('c', 3)
print('LRU keys (newest first):', lru.keys())

**Measure it.** Predict the chart first, then run.

*What to look for:* The sentinel-list LRU stays flat as capacity grows; the list-based LRU grows linearly. OrderedDict (C) is the fastest constant.

In [ ]:
rows_E22 = experiment('circular_doubly')

## E23 — Ring buffer (circular array)

**Concept.** front and back indices wrap with `% capacity`; no element ever moves. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-10--queues-deques-and-ring-buffers-circular-arrays).

**See it** (run the next cell).

In [ ]:
rb = dsa.RingBuffer(8)
for v in range(10, 16): rb.enqueue(v)
for _ in range(4): rb.dequeue()
for v in range(16, 20): rb.enqueue(v)
show('ring-buffer', (rb.block, rb.front, rb.count), f'front={rb.front}, count={rb.count}, wraps={rb.wraps}')
print('queue order:', rb.to_list())

**Measure it.** Predict the chart first, then run.

*What to look for:* Find the crossover point where `list.pop(0)` becomes slower than the Python ring buffer.

In [ ]:
rows_E23 = experiment('ring_buffer')

## E24 — Heapify vs n pushes

**Concept.** Bottom-up heapify is O(n); n pushes are O(n log n). [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-15--binary-heaps-priority-queues-and-heap-sort).

**See it** (run the next cell).

In [ ]:
h = dsa.MinHeap([5, 3, 8, 1, 9, 2])
print('array:', h.a, '| is heap:', h.is_heap(), '| swaps used:', h.swaps)
print('pops:', [h.pop() for _ in range(6)])

**Measure it.** Predict the chart first, then run.

*What to look for:* Heapify swaps stay below n; push swaps grow faster than n.

In [ ]:
rows_E24 = experiment('heap')

## E25 — Trie prefix search

**Concept.** A prefix query walks only the prefix's characters. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-16--tries-prefix-trees).

**See it** (run the next cell).

In [ ]:
t = dsa.Trie(['cat', 'car', 'cart', 'dog'])
print(t.starts_with('ca'), '| contains ca?', t.contains('ca'), '| nodes:', t.node_count())

**Measure it.** Predict the chart first, then run.

*What to look for:* The trie walk is constant (3 characters) while the scan grows with n.

In [ ]:
rows_E25 = experiment('trie')

## E26 — BST deletion

**Concept.** Three cases: leaf, one child, two children (replace by inorder successor). [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-13--binary-search-trees-search-insert-and-the-three-deletion-cases).

**See it** (run the next cell).

In [ ]:
root = None
for k in (8, 4, 12, 2, 6, 10, 14): root = dsa.bst_insert_node(root, k)
root = dsa.bst_delete(root, 12)
dsa.draw_tree(root, title='after deleting 12 (two children): 14 takes its place'); plt.show()

**Measure it.** Predict the chart first, then run.

*What to look for:* Random-order plain BSTs are taller than AVL trees (about 2–3 × log₂ n vs ≤ 1.45 log₂ n).

In [ ]:
rows_E26 = experiment('bst_delete')

## E27 — Union-find

**Concept.** Union by rank and path compression keep trees almost flat. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-18--shortest-paths-topological-order-union-find-and-spanning-trees).

**See it** (run the next cell).

In [ ]:
uf = dsa.UnionFind(6)
for a, b in ((0, 1), (1, 2), (3, 4)): uf.union(a, b)
print('components:', uf.components, '| parents:', uf.parent)

**Measure it.** Predict the chart first, then run.

*What to look for:* Note the log scale: naive is quadratic, the optimised versions are linear in total.

In [ ]:
rows_E27 = experiment('union_find')

## E28 — Topological sort

**Concept.** Kahn removes in-degree-0 vertices; DFS reverses finishing order. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-18--shortest-paths-topological-order-union-find-and-spanning-trees).

**See it** (run the next cell).

In [ ]:
dag = {'accept': ['cook'], 'cook': ['pack', 'notify'], 'pack': ['deliver'], 'notify': [], 'deliver': []}
print(dsa.topo_sort_kahn(dag)[0])

**Measure it.** Predict the chart first, then run.

*What to look for:* Both process every edge exactly once (the three lines coincide).

In [ ]:
rows_E28 = experiment('topo_sort')

## E29 — Open addressing vs chaining

**Concept.** Linear probing degrades sharply as α → 1; chaining degrades gently. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-11--hash-tables-chaining-open-addressing-load-factor-and-resizing).

**See it** (run the next cell).

In [ ]:
oa = dsa.OpenAddressHash(8, max_load=None)
for k in (1, 9, 17): oa.put(k, k)
print('slots:', ['·' if k is None else k for k in oa.keys], '| probes to find 17:', oa.get(17)[1])

**Measure it.** Predict the chart first, then run.

*What to look for:* The measured probing curve follows ½(1 + 1/(1−α)); chaining stays near 1 + α/2.

In [ ]:
rows_E29 = experiment('open_addressing')

## E30 — Dynamic array growth

**Concept.** Doubling copies < 2n elements in total; a fixed increment copies ~n²/(2·increment). [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/DATA_STRUCTURES_ALGORITHMS.md#lesson-4--arrays-dynamic-arrays-multi-dimensional-indexing-and-strings).

**See it** (run the next cell).

In [ ]:
arr = dsa.DynamicArray(2.0)
for v in range(9):
    arr.append(v); print(f'append {v}: size={arr.size:<2} capacity={arr.capacity:<2} copies so far={arr.copies}')

**Measure it.** Predict the chart first, then run.

*What to look for:* Doubling and ×1.5 are straight lines parallel to n on the log plot (linear); +16 is quadratic.

In [ ]:
rows_E30 = experiment('dynamic_array')

In [ ]:
import shutil
dsa.save_diagrams(OUT / 'diagrams')
shutil.make_archive('dsa_results', 'zip', OUT)
print('dsa_results.zip: 30 CSV + 30 PNG charts + structure diagrams')

## Lab report
For each experiment: (1) question, (2) predicted bound, (3) input sizes and distribution, (4) exact counted operation or timing method, (5) chart and CSV, (6) derivation, (7) where the measurement disagrees with theory and why, (8) one extension. Extensions to try: 10 seeds with error bars; full AVL deletion by hand; overwrite-when-full ring buffer; quadratic probing; a heap-based dispatch simulator.